# SE BGC - Poster Results

Mean chlorophyll-a concentration over time by polarity from the SDP pigment results.

In [ ]:
import sys
from typing import cast
import datetime as dt
from pathlib import Path

ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from utils.config import resolve_output_dir

EXPERIMENT = "bgc_20241001_20250701"

pig_frames = []
for polarity in ("cyclone", "anticyclone"):
    pig_dir = resolve_output_dir(EXPERIMENT, "pigments", polarity)
    for fp in sorted(pig_dir.glob("eddy_*_pigments.parquet")):
        df = pd.read_parquet(fp)
        df["polarity"] = polarity
        pig_frames.append(df)
pigments = pd.concat(pig_frames, ignore_index=True)

pig_medians = (
    pigments.groupby(["track_id", "date", "polarity"])["T chla"]
    .mean().reset_index()
)
pig_medians = pig_medians[pig_medians["T chla"] >= 0.01].copy()

print(f"pigment_eddy_dates_t_chla_gte_0_01: {len(pig_medians):,}")


## Plot: Mean Chl-a Over Time by Polarity

Per-eddy-halfmonth means aggregated across eddies (one mean per polarity per half-month bin) with 95% bootstrap CIs (2000 resamples). Colored dots: individual eddy-date spatial means, positioned by actual observation date with low alpha so dense regions cumulatively darken. The single anticyclone June outlier (~2.2 mg/m³) is filtered.

In [ ]:
from calendar import monthrange

pig_m = pig_medians.copy()
pig_m["month"] = pig_m["date"].dt.month  # pyright: ignore[reportAttributeAccessIssue]
pig_m["half"] = (pig_m["date"].dt.day > 15).astype(int)  # pyright: ignore[reportAttributeAccessIssue]

# Per-eddy-halfmonth means for the line + CI computation
eddy_hm = (
    pig_m.groupby(["track_id", "polarity", "month", "half"])["T chla"]
    .mean().reset_index()
)

# Remove the anticyclone June outlier (single ~2.2 mg/m^3 point)
eddy_hm = eddy_hm[
    ~((eddy_hm["polarity"] == "anticyclone") & (eddy_hm["T chla"] > 1.5))
].reset_index(drop=True)

MONTH_ORDER = [10, 11, 12, 1, 2, 3, 4, 5, 6]
MONTH_LABELS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "Jun"]

bins_sequence = [(m, h) for m in MONTH_ORDER for h in (0, 1)]
month_tick_positions = [i * 2 + 0.5 for i in range(len(MONTH_ORDER))]


def map_date_to_x(ts):
    """
    Map a date to a continuous x position on the half-month axis. Each bin occupies [center - 0.5, center + 0.5]; the fractional offset within the bin reflects the actual day of month.
    """
    if ts.month not in MONTH_ORDER:
        return np.nan
    month_idx = MONTH_ORDER.index(ts.month)
    if ts.day <= 15:
        bin_center = month_idx * 2
        bin_start, bin_end = 1, 15
    else:
        bin_center = month_idx * 2 + 1
        bin_start = 16
        bin_end = monthrange(ts.year, ts.month)[1]
    within = (ts.day - bin_start) / (bin_end - bin_start) if bin_end > bin_start else 0.5
    return bin_center - 0.5 + within


# Per-eddy-date scatter positioned by actual observation date
scatter_data = pig_medians.copy()
scatter_data["x"] = scatter_data["date"].apply(map_date_to_x)  # pyright: ignore[reportAttributeAccessIssue]
scatter_data = scatter_data.dropna(subset=["x"])  # pyright: ignore[reportCallIssue]
scatter_data = scatter_data[
    ~((scatter_data["polarity"] == "anticyclone") & (scatter_data["T chla"] > 1.5))
]

fig, axes = cast("tuple[Figure, np.ndarray]", plt.subplots(1, 2, figsize=(12, 5.5), sharey=True,
    gridspec_kw={"wspace": 0.08}))

summaries = {}
for pol in ["cyclone", "anticyclone"]:
    means, lo_all, hi_all, ns = [], [], [], []
    for m, h in bins_sequence:
        vals = eddy_hm.loc[
            (eddy_hm["polarity"] == pol)
            & (eddy_hm["month"] == m)
            & (eddy_hm["half"] == h),
            "T chla",
        ].to_numpy()
        if len(vals) == 0:
            means.append(np.nan); lo_all.append(np.nan); hi_all.append(np.nan); ns.append(0)
            continue
        if len(vals) >= 3:
            rng = np.random.default_rng(42)
            boots = np.array([
                np.mean(rng.choice(vals, size=len(vals), replace=True))
                for _ in range(2000)
            ])
            lo, hi = np.percentile(boots, [2.5, 97.5])
        else:
            lo, hi = np.min(vals), np.max(vals)
        means.append(np.mean(vals))
        lo_all.append(lo); hi_all.append(hi); ns.append(len(vals))
    summaries[pol] = (np.array(means), np.array(lo_all), np.array(hi_all), ns)

panel_info = [
    ("cyclone", "#2166ac"),
    ("anticyclone", "#b2182b"),
]

for ax, (pol, color) in zip(axes, panel_info):
    x_positions = np.arange(len(bins_sequence))

    # Scatter at actual date positions (one dot per per-eddy-date observation); low alpha so overlapping observations cumulatively darken dense regions
    sub = scatter_data[scatter_data["polarity"] == pol]
    ax.scatter(sub["x"], sub["T chla"], s=22, color=color, alpha=0.18,
        linewidths=0, zorder=1)

    means, lo_all, hi_all, ns = summaries[pol]
    valid = ~np.isnan(means)
    # lower row then upper row, the two-row layout errorbar expects: two (n_valid_bins,) -> (2, n_valid_bins)
    yerr = np.vstack([means[valid] - lo_all[valid], hi_all[valid] - means[valid]])
    ax.errorbar(x_positions[valid], means[valid], yerr=yerr, fmt="o-",
        color=color, lw=2.8, ms=8,
        capsize=5, capthick=1.8, elinewidth=1.8,
        markeredgecolor="white", markeredgewidth=0.8,
        zorder=3)

    ax.set_xticks(month_tick_positions)
    ax.set_xticklabels(MONTH_LABELS, fontsize=11)
    ax.set_xlabel("Month", fontsize=12, labelpad=8)
    ax.set_title(pol.capitalize(),
        fontsize=13, fontweight="bold", color=color, pad=12)
    ax.tick_params(axis="both", which="major", labelsize=10, length=4, width=1.0)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_linewidth(1.0)
    ax.grid(axis="y", alpha=0.22, linestyle="--", linewidth=0.6)
    ax.set_axisbelow(True)

y_max = max(eddy_hm["T chla"].max(), scatter_data["T chla"].max()) * 1.06
axes[0].set_ylim(0, y_max)
axes[0].set_ylabel("Mean T chl-a (mg m$^{-3}$)", fontsize=12, labelpad=10)

fig.suptitle("Mean Chl-a Over Time by Polarity",
    fontsize=15, y=1.00, fontweight="bold")
fig.tight_layout()
fig.savefig("3_monthly_tchla_by_polarity.png", dpi=300, bbox_inches="tight")
plt.show()